In [1]:
!pip install folium pandas -q  # Remove -q if you want to see output

import folium
import pandas as pd
from folium.plugins import MarkerCluster
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

# Load the SpaceX dataset
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
spacex_df = pd.read_csv(URL)

# Prepare launch sites dataframe (unique sites with coordinates)
launch_sites_df = spacex_df[['Launch Site', 'Lat', 'Long']].groupby('Launch Site').first().reset_index()

# -----------------------------
# TASK 1: Mark all launch sites on the map
# -----------------------------

# Create map centered to show all US launch sites
site_map = folium.Map(location=[34.0, -100.0], zoom_start=5)

# Add circle + labeled marker for each launch site
for _, site in launch_sites_df.iterrows():
    coordinate = [site['Lat'], site['Long']]
    name = site['Launch Site']
    
    # Orange circle with popup
    folium.Circle(
        location=coordinate,
        radius=1000,
        color='#d35400',
        fill=True,
        fill_color='#d35400',
        fill_opacity=0.4
    ).add_child(folium.Popup(name)).add_to(site_map)
    
    # Bold text label directly on map
    folium.Marker(
        location=coordinate,
        icon=DivIcon(
            icon_size=(150,36),
            icon_anchor=(0,0),
            html=f'<div style="font-size: 14px; font-weight: bold; color: #d35400;">{name}</div>'
        )
    ).add_to(site_map)

# -----------------------------
# TASK 2: Mark success/failed launches with colored markers
# -----------------------------

# Create marker cluster
marker_cluster = MarkerCluster().add_to(site_map)

# Create marker_color column: green for success (class=1), red for failure (class=0)
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else 'red')

# Add a marker for every launch
for idx, row in spacex_df.iterrows():
    folium.Marker(
        location=[row['Lat'], row['Long']],
        popup=folium.Popup(f"{row['Launch Site']}<br>Success: {'Yes' if row['class']==1 else 'No'}"),
        icon=folium.Icon(color='white', icon_color=row['marker_color'])
    ).add_to(marker_cluster)

# Add MousePosition plugin for easy coordinate reading
from folium.plugins import MousePosition
MousePosition(
    position='topright',
    separator=' | ',
    prefix='Coordinates:'
).add_to(site_map)

# -----------------------------
# TASK 3: Calculate and mark distances to proximities (example: coastline)
# -----------------------------

def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c

# Example: Distance from CCAFS SLC-40 to closest coastline point
launch_coord = [28.562302, -80.577356]  # CCAFS SLC-40
coastline_coord = [28.56398, -80.56802]  # Closest coastline point (you can adjust)

distance_coastline = calculate_distance(*launch_coord, *coastline_coord)

# Marker showing distance
folium.Marker(
    coastline_coord,
    icon=DivIcon(
        icon_size=(150,36),
        icon_anchor=(0,0),
        html=f'<div style="font-size: 12px; color:#d35400;"><b>{distance_coastline:.2f} KM</b></div>'
    )
).add_to(site_map)

# PolyLine connecting launch site to coastline
folium.PolyLine(
    locations=[launch_coord, coastline_coord],
    weight=2,
    color='blue',
    opacity=0.8
).add_to(site_map)

# Optional: You can repeat the above for railway, highway, city, etc.

# -----------------------------
# Display the final interactive map
# -----------------------------
site_map